# Scaling Laws — Scaling Laws for Neural Language Models (2020)

_Papers · Transformers & LLMs_

**Test loss falls as a smooth power law in model size, data, and compute — so you can predict it.**

---

This notebook is a practice scaffold for the **Scaling Laws — Scaling Laws for Neural Language Models (2020)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — NumPy

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# (1) WORKED EXAMPLE: the paper's transcribed law L(N) = (N_c / N) ** alpha_N
#     with the FITTED constants from Table 5. We verify the "10x -> x0.84" drop.
# ---------------------------------------------------------------------------
N_c, alpha_N = 8.8e13, 0.076          # paper's fitted values (Table 5)
def L(N): return (N_c / N) ** alpha_N

L_small = L(1e6)                       # a 1-million-parameter model
L_big   = L(1e7)                       # 10x bigger
print("L(1e6) = %.3f nats" % L_small)
print("L(1e7) = %.3f nats" % L_big)
print("ratio  = %.3f   (shortcut 10**-alpha_N = %.3f)" % (
      L_big / L_small, 10 ** (-alpha_N)))
# L(1e6) = 4.016 nats
# L(1e7) = 3.371 nats
# ratio  = 0.839   (shortcut 10**-alpha_N = 0.839)

# ---------------------------------------------------------------------------
# (2) CONCEPTUAL FIT: synthetic (N, loss) points that FOLLOW a power law, then
#     recover its slope by least-squares in log-log space. ROUND, MADE-UP
#     constants -- an illustration of the FORM, NOT the paper's measured values.
# ---------------------------------------------------------------------------
rng = np.random.default_rng(0)
Nc_demo, a_demo = 1.0e9, 0.10          # synthetic constants (not the paper's)
N_pts = np.array([1e5, 1e6, 1e7, 1e8, 1e9])          # five model sizes
loss  = (Nc_demo / N_pts) ** a_demo
loss *= rng.normal(1.0, 0.01, size=loss.shape)       # tiny jitter, so the fit works

# A power law is a line in log-log space: ln(loss) = ln(Nc^a) - a*ln(N).
slope, intercept = np.polyfit(np.log(N_pts), np.log(loss), 1)
print("\nsynthetic power-law fit (illustration of FORM only):")
print("  recovered slope  = %.4f  (true -alpha = %.4f)" % (slope, -a_demo))
print("  recovered alpha  = %.4f  (true alpha  = %.4f)" % (-slope, a_demo))
# synthetic power-law fit (illustration of FORM only):
#   recovered slope  = -0.1005  (true -alpha = -0.1000)
#   recovered alpha  =  0.1005  (true alpha  =  0.1000)
# The straight log-log line IS the power law. These constants are synthetic,
# NOT the paper's measured exponents.

## Visualize the data & results

_A power law L = (N_c/N)^alpha plots as a STRAIGHT line on log-log axes. If we generate a few synthetic (model size, loss) points from such a law and least-squares-fit them in log-log space, do we recover the line?_

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# Synthetic power law: L = (N_c / N) ** alpha. ROUND, MADE-UP constants --
# this illustrates the FORM (a straight log-log line), NOT the paper's values.
N_c, alpha = 1.0e9, 0.10
N = np.array([1e5, 1e6, 1e7, 1e8, 1e9])
loss = (N_c / N) ** alpha
loss *= rng.normal(1.0, 0.01, size=loss.shape)        # 1% jitter

# Fit a line in log-log space; slope is -alpha.
x, y = np.log10(N), np.log10(loss)
slope, intercept = np.polyfit(x, y, 1)
print("data points (log10 N, log10 loss):")
for xi, yi in zip(x, y):
    print("  %.1f  %.4f" % (xi, yi))
xs = np.array([x.min(), x.max()])
print("fit line endpoints:", list(zip(xs.round(1), (slope*xs + intercept).round(4))))
print("recovered slope = %.4f (= -alpha), so alpha = %.4f" % (slope, -slope))
# data points (log10 N, log10 loss):
#   5.0  0.4005
#   6.0  0.2994
#   7.0  0.2028
#   8.0  0.1005
#   9.0  -0.0023
# fit line endpoints: [(5.0, 0.4011), (9.0, -0.0008)]
# recovered slope = -0.1005 (= -alpha), so alpha = 0.1005
# A straight log-log line IS a power law. Synthetic constants, NOT the paper's.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Per-decade loss drop. The paper fits $L(N) = (N_c/N)^{\alpha_N}$ with
            $\alpha_N \approx 0.076$. (a) By what factor does the loss change if you make the model
            $1000\times$ bigger? (b) If a different resource had exponent $0.095$ (the paper's $\alpha_D$),
            which scales loss down faster per decade, size or data?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use the per-decade factor $10^{-\alpha}$. One decade ($10\times$) of size: $10^{-0.076} \approx 0.839$. — _A power law in $N$ means a $10\times$ increase multiplies the loss by $10^{-\alpha_N}$, independent of where you start._
- $1000\times = 10^3$ is three decades, so multiply three factors: $0.839^3 \approx 0.590$. — _Each decade applies the same multiplicative factor; three decades compound it three times._
- Compare exponents: $10^{-0.095} \approx 0.804$ per decade for data vs. $0.839$ for size. The larger exponent gives the smaller multiplier, hence the faster drop. — _A bigger exponent is a steeper log-log slope, so loss falls faster per decade of that resource._

**Answer:** (a) $1000\times$ bigger is three decades: $0.839^3 \approx 0.59$, about a 41% loss
                 reduction. (b) Data, with exponent $\alpha_D \approx 0.095$, drops the loss faster per
                 decade ($\times 0.804$) than size with $\alpha_N \approx 0.076$ ($\times 0.839$) &mdash; a
                 bigger exponent is a steeper log-log slope. (Both still require many orders of magnitude to move
                 the loss a lot, which is the whole point: smooth, slow, predictable.)

</details>

**Problem 2.** Reading the log-log line. Someone hands you a log-log plot (natural log of loss on the
            vertical axis, natural log of $N$ on the horizontal) and the points lie on a straight line of slope
            $-0.076$ passing through a known point. What does the slope tell you, what would a steeper
            (more negative) slope mean, and how would you read off the scale constant $N_c$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall $\ln L = \alpha_N \ln N_c - \alpha_N \ln N$, a line with slope $-\alpha_N$. — _Taking the log of $L = (N_c/N)^{\alpha_N}$ turns the power law into a linear equation in $\ln N$._
- Identify the slope: $-0.076$ means $\alpha_N = 0.076$. A steeper (more negative) slope means a larger $\alpha_N$, so loss falls faster per decade of size. — _The slope of the log-log line IS the negative exponent; steeper means a bigger exponent._
- Find $N_c$: it is the $N$ where the line predicts $L = 1$ (since $\ln L = 0$ when $N = N_c$). Read the horizontal intercept at $\ln L = 0$, then exponentiate. — _At $N = N_c$ the ratio $N_c/N = 1$, so $L = 1^{\alpha_N} = 1$ and $\ln L = 0$ &mdash; the line crosses zero exactly at $\ln N_c$._

**Answer:** The slope $-0.076$ is $-\alpha_N$: it sets how fast loss falls per decade of model size
                 ($10^{-0.076}\approx 0.84$ per $10\times$). A steeper, more-negative slope means a larger
                 exponent and a faster per-decade drop. The scale constant $N_c$ is the model size at which the
                 fitted line predicts $L = 1$ (where $\ln L = 0$), because there $N_c/N = 1$. So the slope gives
                 the exponent and the zero-crossing gives the constant &mdash; the two numbers in Table 5.

</details>

**Problem 3.** The budget rule (compute allocation). You double your training compute budget $C$. Using the
            paper's compute-optimal scaling $N \propto C^{0.73}$, by roughly what factor should the optimal model
            size grow? And what does the paper say to do about how long you train? Tie it to "stop before
            convergence."

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply the scaling: doubling $C$ multiplies optimal $N$ by $2^{0.73}$. — _$N \propto C^{0.73}$ (Table 6) means the optimal model size is compute raised to the $0.73$ power, so a $\times 2$ budget gives $\times 2^{0.73}$ size._
- Compute $2^{0.73} = e^{0.73 \ln 2} = e^{0.506} \approx 1.66$. So optimal $N$ grows about $1.66\times$. — _Most of the doubled budget goes into a bigger model, not proportionally more training tokens._
- Since $C \approx 6NBS$ and $N$ absorbs most of the increase, the number of tokens (steps) grows much less than $2\times$; you stop training the bigger model relatively early. — _The paper's finding: larger models are more sample-efficient, so optimal training stops "significantly before convergence."_

**Answer:** Doubling compute should grow the optimal model size by about $2^{0.73} \approx 1.66\times$ &mdash;
                 most of the extra budget buys a bigger model, not proportionally more data. Because that
                 larger model is more sample-efficient and compute is split as $C \approx 6NBS$, you process
                 relatively few extra tokens and stop before convergence. That is the abstract's recipe:
                 "training very large models on a relatively modest amount of data and stopping significantly
                 before convergence." (Quoted, abstract.)

</details>